# Injection-recovery of HARPS-N − SunSim with the criteria of the machine-learning studies

Same data as `injection_baluev_no_fit`: HARPS-N and HARPS-N − SunSim with the median of every one of the **21
intervals** removed. Model at every trial period: a **constant** + the planet sinusoid (least squares, equal weights), no
offset or other signal fitted. The planet is injected in the RVs at the HARPS-N days before the fit.

Each section repeats the test with the criterion of one paper, as described in its text:

| | data | injections | detection |
|---|---|---|---|
| Zhao et al. (2024) | HARPS-N 2015–2024, daily | P log-spaced 10–550 d, K 0.1–0.5 m/s (step 0.033), one injection per (P, K) | power at the injected period > 0.1 % FAP ("numerically estimated"); limit = lowest K detected |
| Colwell et al. (2025) | HARPS-N 2015–2018 (3 yr), exposures | P = 50 d, K = 0.1–1.0 m/s; P = 10–250 d at K = 1 m/s | period of the highest peak and amplitude fitted, compared with the injected ones (no FAP) |
| Gómez-Vargas et al. (2026) | HARPS-N 2015–2024, daily | P 10–550 d, 10 realisations | power within ±5 % of the injected period > 0.1 % FAP in ≥ 70 % of the realisations |
| Liang et al. (2026) | NEID 2020–2024, 1294 d | 500 trials, P log-uniform 2.5–400 d, K uniform 0.1–0.7 m/s, random phase | highest candidate, threshold raised until no false positive among the 500 trials; recovered if its period is within 20 % and \|K_rec − K\|/K < 1 |

Choices where the papers do not say: the 0.1 % FAP of Zhao is a bootstrap (values shuffled within the intervals), the
period tolerance of Zhao is ±5 % (as Gómez-Vargas), the FAP of Gómez-Vargas is analytic, the success of Colwell is the
highest peak within 5 % of the period and the amplitude within 20 %. The data differ from Colwell (daily averages here)
and Liang (HARPS-N 2015–2024 here, NEID there).

In [1]:
import os

import numpy as np
import matplotlib.pyplot as plt

from sunsim import local_path, observations as obs
from sunsim.residuals import ResidualAnalysis

SIM_RV = os.path.join(local_path('personal_data'), 'rv-15.txt')
ORIGIN = '2015-07-18'
FIRST, LAST = '2015-07-18', '2024-01-28'
FIGURES = "/Users/sophiestucki/Documents/rv_opt/figures/final/"
FAP = 1e-3
SEED = 1
N_NULL = 2000                        # noise versions for the empirical FAPs
N_PHASES = 100                       # phases per (P, K) where a fraction of detections is computed


def save(fig, name):
    if FIGURES:
        fig.savefig(os.path.join(FIGURES, name + '.pdf'), bbox_inches='tight', dpi=300)

In [2]:
t_s, rv_s = np.loadtxt(SIM_RV).T
window = (t_s >= obs.jd_of_date(FIRST) - obs.jd_of_date(ORIGIN)) & (t_s < obs.jd_of_date(LAST) - obs.jd_of_date(ORIGIN) + 1)
t_s, rv_s = t_s[window], rv_s[window]
harps = obs.load_harpsn(local_path('harpsn_sun'), ORIGIN, end=LAST, t_start=FIRST)
cuts = obs.load_interventions(local_path('harpsn_interventions'), ORIGIN)
rv = obs.compare_rv(harps, t_s, rv_s, interventions=cuts)
obs.print_summary(rv=rv)

  period  0:    11.5 ->    87.0  sim   75  obs   53  matched   52  shift    1  RMS 0.850 m/s
  period  1:    87.0 ->   257.0  sim  168  obs  103  matched  101  shift    0  RMS 0.819 m/s
  period  2:   257.0 ->   466.0  sim  206  obs  172  matched  170  shift    1  RMS 0.743 m/s
  period  3:   466.0 ->   633.0  sim  166  obs   86  matched   85  shift    1  RMS 0.663 m/s
  period  4:   633.0 ->   850.0  sim  215  obs  161  matched  159  shift    1  RMS 0.711 m/s
  period  5:   850.0 ->  1010.0  sim  159  obs   46  matched   45  shift   -1  RMS 0.729 m/s
  period  6:  1010.0 ->  1063.0  sim   53  obs   39  matched   38  shift    1  RMS 0.492 m/s
  period  7:  1063.0 ->  1191.0  sim  126  obs  105  matched  102  shift    1  RMS 0.689 m/s
  period  8:  1191.0 ->  1333.0  sim  141  obs  107  matched  105  shift    1  RMS 0.751 m/s
  period  9:  1333.0 ->  1462.0  sim  129  obs  109  matched  108  shift    1  RMS 0.623 m/s
  period 10:  1462.0 ->  1618.0  sim  156  obs   91  matched   91  shi

## Model: constant (or offsets + nuisance sinusoids) fitted with the planet

In [3]:
class NuisanceResidualAnalysis(ResidualAnalysis):
    """GLS with nuisance terms fitted together with the sinusoid at every trial frequency (weighted least squares):
    one offset per interval (groups; offsets=False: one constant) and a free sinusoid at every period of
    nuisance_periods [days]. power = chi2 reduction relative to the nuisance-only fit."""

    def __init__(self, *args, nuisance_periods=(), offsets=True, **kw):
        super().__init__(*args, **kw)
        self.offsets = offsets
        if offsets:
            g = np.unique(self.groups, return_inverse=True)[1]
            X = np.zeros((self.N, int(g.max()) + 1))
            X[np.arange(self.N), g] = 1
        else:
            X = np.ones((self.N, 1))
        self.n_groups = X.shape[1]
        self.nuisance_periods = tuple(nuisance_periods)
        cols = [X] + [np.column_stack([np.cos(2 * np.pi * self.th / P), np.sin(2 * np.pi * self.th / P)])
                      for P in self.nuisance_periods]
        self.X = np.hstack(cols)
        self.n_nuisance = self.X.shape[1]
        self._WX = self.X * self.w[:, None]
        self._Hinv = np.linalg.inv(self.X.T @ self._WX)
        nf = len(self.freq)
        CC, SS, CS = np.empty(nf), np.empty(nf), np.empty(nf)
        for sl, cos, sin in self._trig():
            Ac, As = cos @ self._WX, sin @ self._WX
            CC[sl] = (cos * cos) @ self.w - np.einsum('fq,qr,fr->f', Ac, self._Hinv, Ac)
            SS[sl] = (sin * sin) @ self.w - np.einsum('fq,qr,fr->f', As, self._Hinv, As)
            CS[sl] = (cos * sin) @ self.w - np.einsum('fq,qr,fr->f', Ac, self._Hinv, As)
        self._CC, self._SS, self._CS = CC, SS, CS
        self._D = CC * SS - CS ** 2
        self._C, self._S = np.zeros(nf), np.zeros(nf)

    def center(self, y):
        y = np.asarray(y, dtype=np.float64)
        return y - self.X @ (self._Hinv @ (self._WX.T @ y))

    def _gls(self, y):
        y = np.asarray(y, dtype=np.float64)
        one = y.ndim == 1
        y = y[:, None] if one else y
        wy = self.center(y)
        YY = self.w @ wy ** 2
        wy = wy * self.w[:, None]
        YC, YS = np.empty((len(self.freq), y.shape[1])), np.empty((len(self.freq), y.shape[1]))
        for sl, cos, sin in self._trig():
            YC[sl], YS[sl] = cos @ wy, sin @ wy
        CC, SS, CS, D = (v[:, None] for v in (self._CC, self._SS, self._CS, self._D))
        a = (YC * SS - YS * CS) / D
        b = (YS * CC - YC * CS) / D
        power = (SS * YC * YC + CC * YS * YS - 2 * CS * YC * YS) / (YY * D)
        out = (power, a, b, np.zeros_like(a))
        return tuple(v[:, 0] for v in out) if one else out

    def fap(self, power):
        prob = (1 - np.asarray(power, dtype=np.float64)) ** ((self.N - self.n_nuisance - 2) / 2)
        Mprob = self.M * prob
        out = np.where(Mprob > 0.01, 1 - (1 - prob) ** self.M, Mprob)
        return out if out.ndim else float(out)

    def power_level(self, fap):
        prob = 1 - (1 - np.asarray(fap, dtype=np.float64)) ** (1 / self.M)
        return 1 - prob ** (2 / (self.N - self.n_nuisance - 2))

    def subset(self, keep, **kw):
        """The same model on the points keep (boolean), e.g. a shorter time span."""
        return NuisanceResidualAnalysis(self.t[keep], self.y[keep], groups=self.groups[keep],
                                        t_planet=self.t_planet[keep], nuisance_periods=self.nuisance_periods,
                                        offsets=self.offsets, **kw)


def merge_offsets(ra, nuisance_periods=(), verbose=True):
    """Greedy merging of neighbouring intervals while it lowers the BIC of the fit of ra.y by one offset per interval +
    the nuisance sinusoids. Returns the new group of every point."""
    g = np.unique(ra.groups, return_inverse=True)[1]
    extra = [np.column_stack([np.cos(2 * np.pi * ra.th / P), np.sin(2 * np.pi * ra.th / P)]) for P in nuisance_periods]

    def rss(groups):
        X = np.zeros((ra.N, int(groups.max()) + 1))
        X[np.arange(ra.N), groups] = 1
        X = np.hstack([X] + extra)
        r = ra.y - X @ np.linalg.lstsq(X, ra.y, rcond=None)[0]
        return r @ r
    while g.max() > 0:
        base = rss(g)
        dbic, b = min((ra.N * np.log(rss(np.where(g > b, g - 1, g)) / base) - np.log(ra.N), b) for b in range(g.max()))
        if dbic >= 0:
            break
        g = np.where(g > b, g - 1, g)
    if verbose:
        print('%d offsets kept out of %d' % (g.max() + 1, len(np.unique(ra.groups))))
    return g

In [4]:
# the 21 interval medians are removed from both series (compare_rv for 'res', period_medians for 'obs'); groups are
# kept for the bootstrap inside the intervals
models = {key: NuisanceResidualAnalysis.from_comparison(rv, key, period_medians=(key == 'obs'), offsets=False,
                                                        name=name)
          for key, name in (('res', 'HARPS-N - SunSim'), ('obs', 'HARPS-N'))}
print('constant only, %d points, %d intervals (medians removed)' % (models['res'].N, len(np.unique(models['res'].groups))))


constant only, 1984 points, 21 intervals (medians removed)


## Tools

In [5]:
def injected(ra, P, K, t0):
    """ra.y + K sin(2 pi (t_planet - t0) / P), one column per (P, K, t0) (arrays of the same length or scalars)."""
    P, K, t0 = np.broadcast_arrays(np.atleast_1d(P), np.atleast_1d(K), np.atleast_1d(t0))
    return ra.y[:, None] + K[None] * np.sin(2 * np.pi * (ra.t_planet[:, None] - t0[None]) / P[None])


def analyse(ra, Y, P_inj, tol=0.05, chunk=200):
    """For every column of Y: period, power and semi-amplitude of the highest peak, and the highest power within
    ±tol of the injected period P_inj (one per column)."""
    P_inj = np.broadcast_to(np.atleast_1d(P_inj), (Y.shape[1],))
    out = {k: np.empty(Y.shape[1]) for k in ('P', 'pmax', 'K', 'local')}
    cols = np.arange(Y.shape[1])
    for i in range(0, Y.shape[1], chunk):
        sl = slice(i, i + chunk)
        power, a, b, _ = ra._gls(Y[:, sl])
        k = power.argmax(axis=0)
        c = np.arange(power.shape[1])
        out['P'][sl], out['pmax'][sl], out['K'][sl] = 1 / ra.freq[k], power[k, c], np.hypot(a[k, c], b[k, c])
        f = ra.freq[:, None]
        near = (f >= 1 / ((1 + tol) * P_inj[sl])) & (f <= 1 / ((1 - tol) * P_inj[sl]))
        out['local'][sl] = np.where(near, power, -np.inf).max(axis=0)
    return out


def null_level(ra, method='shuffle', fap=FAP, n=N_NULL, seed=SEED):
    """Power of the highest peak exceeded by a fraction fap of the noise-only versions: 'shuffle' (values permuted
    inside every interval: bootstrap) or 'shift' (series rolled cyclically: keeps the correlated noise)."""
    rng = np.random.default_rng(seed)
    if method == 'shuffle':
        Y = np.repeat(ra.y[:, None], n, axis=1)
        for g in np.unique(ra.groups):
            idx = np.flatnonzero(ra.groups == g)
            Y[idx] = rng.permuted(Y[idx], axis=0)
    else:
        k = rng.integers(int(0.05 * ra.N), int(0.95 * ra.N), size=n)
        Y = np.stack([np.roll(ra.y, s) for s in k], axis=1)
    return np.quantile(analyse(ra, Y, np.ones(n))['pmax'], 1 - fap)


def limit(K, rate, level):
    """Lowest K where rate [%] (along K) first reaches level (linear interpolation; NaN: never)."""
    above = np.flatnonzero(rate >= level)
    if len(above) == 0:
        return np.nan
    i = above[0]
    return K[0] if i == 0 else float(np.interp(level, rate[i - 1:i + 1], K[i - 1:i + 1]))


def msini(K, P):
    """M sin i [Earth masses] for K [m/s], P [d], 1 Msun, circular orbit."""
    k_earth = (2 * np.pi * 6.674e-11 / (365.25 * 86400)) ** (1 / 3) * 5.972e24 / 1.989e30 ** (2 / 3)
    return np.asarray(K) / k_earth * (np.asarray(P) / 365.25) ** (1 / 3)

summary = {}

## Zhao et al. (2024)
Grid of their paper (P log-spaced 10–550 d, K 0.1–0.5 m/s by 0.033), power within ±5 % of the injected period above the
bootstrap 0.1 % FAP. Their one injection per (P, K) is replaced by 100 phases: the map is the probability that one
injection is detected, and their "lowest K detected" corresponds to ~50 %.

In [6]:
P_Z = np.geomspace(10, 550, 12)
K_Z = np.arange(0.1, 0.5 + 1e-9, 0.1 / 3)
rate_Z, fp_Z = {}, {}
for key, ra in models.items():
    thr = null_level(ra, 'shuffle')
    fp_Z[key] = np.array([analyse(ra, ra.y[:, None], P)['local'][0] >= thr for P in P_Z])   # passes with no planet
    rate = np.empty((len(K_Z), len(P_Z)))
    for j, P in enumerate(P_Z):
        t0 = ra.t[0] + P * np.arange(N_PHASES) / N_PHASES
        for i, K in enumerate(K_Z):
            rate[i, j] = 100 * np.mean(analyse(ra, injected(ra, P, K, t0), P)['local'] >= thr)
    rate_Z[key] = rate
    lim = [limit(K_Z, rate[:, j], 50) for j in range(len(P_Z))]
    print('%-18s K at 50 %% (* = the noise alone passes, not usable):' % ra.name)
    print('   ' + ' '.join('%4.0f d %.2f%s' % (p, k, '*' if f else ' ') for p, k, f in zip(P_Z, lim, fp_Z[key])))
lims = np.array([limit(K_Z, rate_Z['res'][:, j], 50) for j in range(len(P_Z))])[~fp_Z['res']]
summary['Zhao et al. (2024)'] = ('0.18 m/s (10-550 d)', '%.2f-%.2f m/s (%d of %d periods usable)' % (
    np.nanmin(lims), np.nanmax(lims), len(lims), len(P_Z)))

HARPS-N - SunSim   K at 50 % (* = the noise alone passes, not usable):
     10 d 0.12    14 d 0.10    21 d 0.13    30 d 0.10*   43 d 0.14    62 d 0.13    89 d 0.13   128 d 0.12   184 d 0.12   265 d 0.10   382 d 0.11   550 d 0.10 
HARPS-N            K at 50 % (* = the noise alone passes, not usable):
     10 d 0.23    14 d 0.10*   21 d 0.20    30 d 0.10*   43 d 0.23    62 d 0.19    89 d 0.11   128 d 0.10*  184 d 0.23   265 d 0.14   382 d 0.10   550 d 0.10*


## Colwell et al. (2025)
First 3 years (2015-07-18 to 2018-07-18). Amplitude sweep at P = 50 d (K = 0.1–1.0 m/s) and period sweep at K = 1 m/s
(P = 10–250 d). Recovered: highest peak within 5 % of the period and fitted K within 20 %. Their result: 0.2 m/s at
50 d recovered with 8.8 % error on K and 0.7 % on P.

In [7]:
t_end = obs.jd_of_date('2018-07-18') - obs.jd_of_date(ORIGIN)
ra_c = models['res'].subset(models['res'].t <= t_end, name='HARPS-N - SunSim, 2015-2018')
print('%d days' % ra_c.N)
K_C = np.arange(0.1, 1.0 + 1e-9, 0.1)
t0 = ra_c.t[0] + 50 * np.arange(N_PHASES) / N_PHASES
print('P = 50 d:   K    recovered [%]   median |dK|/K   median |dP|/P')
rec_C = []
for K in K_C:
    r = analyse(ra_c, injected(ra_c, 50.0, K, t0), 50.0)
    ok = (np.abs(r['P'] - 50) / 50 <= 0.05) & (np.abs(r['K'] - K) / K <= 0.2)
    rec_C.append(100 * ok.mean())
    print('          %4.1f  %8.0f        %10.3f      %10.4f' % (K, 100 * ok.mean(), np.median(np.abs(r['K'] - K) / K),
                                                            np.median(np.abs(r['P'] - 50) / 50)))
print('P sweep at K = 1 m/s:', end='')
for P in (10, 25, 40, 50, 75, 100, 150, 200, 250):
    r = analyse(ra_c, injected(ra_c, float(P), 1.0, ra_c.t[0] + P * np.arange(N_PHASES) / N_PHASES), float(P))
    print('  %d d %.0f %%' % (P, 100 * np.mean(np.abs(r['P'] - P) / P <= 0.05)), end='')
print()
summary['Colwell et al. (2025)'] = ('0.2 m/s at 50 d', '%.2f m/s at 50 d (50 %% recovered)' % limit(K_C, np.array(rec_C), 50))

679 days
P = 50 d:   K    recovered [%]   median |dK|/K   median |dP|/P
           0.1         0             0.700          0.7139
           0.2        67             0.130          0.0064
           0.3       100             0.107          0.0018
           0.4       100             0.084          0.0018
           0.5       100             0.069          0.0018
           0.6       100             0.057          0.0018
           0.7       100             0.049          0.0018
           0.8       100             0.043          0.0018
           0.9       100             0.038          0.0018
           1.0       100             0.034          0.0018
P sweep at K = 1 m/s:  10 d 100 %  25 d 100 %  40 d 100 %  50 d 100 %  75 d 100 %  100 d 100 %  150 d 100 %  200 d 100 %  250 d 100 %


## Gómez-Vargas et al. (2026)
P 10–550 d, 10 realisations (random phases) per (P, K), power within ±5 % of the injected period above the analytic
0.1 % FAP in ≥ 70 % of them.

In [8]:
rng = np.random.default_rng(SEED)
K_G = np.arange(0.05, 0.5 + 1e-9, 0.025)
rate_G, fp_G = {}, {}
for key, ra in models.items():
    thr = ra.power_level(FAP)
    fp_G[key] = np.array([analyse(ra, ra.y[:, None], P)['local'][0] >= thr for P in P_Z])
    rate = np.empty((len(K_G), len(P_Z)))
    for j, P in enumerate(P_Z):
        t0 = ra.t[0] + P * rng.random(10)
        for i, K in enumerate(K_G):
            rate[i, j] = 100 * np.mean(analyse(ra, injected(ra, P, K, t0), P)['local'] >= thr)
    rate_G[key] = rate
    lim = [limit(K_G, rate[:, j], 70) for j in range(len(P_Z))]
    print('%-18s K at 70 %% (* = the noise alone passes, not usable):' % ra.name)
    print('   ' + ' '.join('%4.0f d %.2f%s' % (p, k, '*' if f else ' ') for p, k, f in zip(P_Z, lim, fp_G[key])))
lims = np.array([limit(K_G, rate_G['res'][:, j], 70) for j in range(len(P_Z))])[~fp_G['res']]
summary['Gómez-Vargas et al. (2026)'] = ('>= 0.25 m/s (10-550 d)', '%.2f-%.2f m/s (%d of %d periods usable)' % (
    np.nanmin(lims), np.nanmax(lims), len(lims), len(P_Z)))

HARPS-N - SunSim   K at 70 % (* = the noise alone passes, not usable):
     10 d 0.15    14 d 0.09    21 d 0.15    30 d 0.05*   43 d 0.14    62 d 0.12    89 d 0.15   128 d 0.15   184 d 0.12   265 d 0.05   382 d 0.15   550 d 0.05*
HARPS-N            K at 70 % (* = the noise alone passes, not usable):
     10 d 0.23    14 d 0.05*   21 d 0.31    30 d 0.05*   43 d 0.24    62 d 0.36    89 d 0.15   128 d 0.26*  184 d 0.23   265 d 0.30   382 d 0.08   550 d 0.05*


## Liang et al. (2026)
500 trials (P log-uniform 2.5–400 d, K uniform 0.1–0.7 m/s, random phase). The top candidate (highest peak) of each
trial is accepted if its power exceeds a threshold; it is a recovery if its period is within 20 % of the injected one
and |K_rec − K|/K < 1, otherwise a false positive. As in the paper, the threshold is the lowest one that gives no
false positive among the 500 trials. The frequency grid goes down to 2.2 d here.

In [9]:
rng = np.random.default_rng(SEED)
n_tr = 500
P_L = np.exp(rng.uniform(np.log(2.5), np.log(400), n_tr))
K_L = rng.uniform(0.1, 0.7, n_tr)
base = models['res']
ra_l = NuisanceResidualAnalysis(base.t, base.y, groups=base.groups, t_planet=base.t_planet,
                                nuisance_periods=base.nuisance_periods, offsets=base.offsets, fend=1 / 2.2,
                                name='HARPS-N - SunSim')
t0 = ra_l.t[0] + P_L * rng.random(n_tr)
r = analyse(ra_l, injected(ra_l, P_L, K_L, t0), P_L)
good = (np.abs(r['P'] - P_L) / P_L <= 0.2) & (np.abs(r['K'] - K_L) / K_L < 1)
thr = r['pmax'][~good].max() if (~good).any() else 0                   # lowest threshold with no false positive
found = good & (r['pmax'] > thr)
print('threshold power %.4f (analytic FAP %.1e)' % (thr, ra_l.fap(thr)))
print('recovered %d / %d, of which %d with K < 0.3 m/s' % (found.sum(), n_tr, (found & (K_L < 0.3)).sum()))
bins = np.arange(0.1, 0.71, 0.1)
for lo, hi in zip(bins[:-1], bins[1:]):
    m = (K_L >= lo) & (K_L < hi)
    print('  K %.1f-%.1f m/s: %3.0f %% recovered (%d trials)' % (lo, hi, 100 * found[m].mean(), m.sum()))
summary['Liang et al. (2026)'] = ('238 / 500 recovered, 13 with K < 0.3 m/s (NEID, AESTRA)',
                                  '%d / 500 recovered, %d with K < 0.3 m/s' % (found.sum(), (found & (K_L < 0.3)).sum()))

threshold power 0.0263 (analytic FAP 4.9e-09)
recovered 432 / 500, of which 98 with K < 0.3 m/s
  K 0.1-0.2 m/s:  25 % recovered (75 trials)
  K 0.2-0.3 m/s:  87 % recovered (91 trials)
  K 0.3-0.4 m/s: 100 % recovered (82 trials)
  K 0.4-0.5 m/s: 100 % recovered (71 trials)
  K 0.5-0.6 m/s: 100 % recovered (85 trials)
  K 0.6-0.7 m/s: 100 % recovered (96 trials)


## Summary

In [10]:
print('%-28s %-58s %s' % ('criterion of', 'their result', 'HARPS-N - SunSim, 21 medians removed, constant only'))
for name, (theirs, ours) in summary.items():
    print('%-28s %-58s %s' % (name, theirs, ours))

criterion of                 their result                                               HARPS-N - SunSim, 21 medians removed, constant only
Zhao et al. (2024)           0.18 m/s (10-550 d)                                        0.10-0.14 m/s (11 of 12 periods usable)
Colwell et al. (2025)        0.2 m/s at 50 d                                            0.17 m/s at 50 d (50 % recovered)
Gómez-Vargas et al. (2026)   >= 0.25 m/s (10-550 d)                                     0.05-0.15 m/s (10 of 12 periods usable)
Liang et al. (2026)          238 / 500 recovered, 13 with K < 0.3 m/s (NEID, AESTRA)    432 / 500 recovered, 98 with K < 0.3 m/s


In [11]:
ra = models['res']
k = np.random.default_rng(SEED).integers(int(0.05 * ra.N), int(0.95 * ra.N), size=N_NULL)
Y0 = np.stack([np.roll(ra.y, s) for s in k], axis=1)
thr_z, thr_g = null_level(ra, 'shuffle'), ra.power_level(FAP)
print('P [d]          ' + ''.join('%6.0f' % p for p in P_Z))
for name, thr in (('Zhao-like', thr_z), ('Gomez-Vargas-like', thr_g)):
    fa = [100 * np.mean(analyse(ra, Y0, np.full(N_NULL, P))['local'] >= thr) for P in P_Z]
    print('%-15s' % name + ''.join('%6.1f' % v for v in fa))

P [d]              10    14    21    30    43    62    89   128   184   265   382   550
Zhao-like         0.0   3.4   1.9   2.8   0.8   0.0   0.0   0.0   5.8  13.7   4.9  53.6
Gomez-Vargas-like   0.1   5.8   3.0   3.8   1.3   0.0   0.0   0.0   9.2  16.7   7.3  58.1
